# References

- Article: https://realpython.com/python-modules-packages/
- Crash-course: https://www.youtube.com/watch?v=VEbuZox5qC4

---

# TLDR / Summary


Most output has been set to `np.ndarrays`, `np.float64` and `np.int64` so that in future, passing exact `dtype` to Numba is easier.

1. **Set modules PATH**
    - Enable access to modules within `utils` package.
    - Able to access files or directories easier, with `project_root`.

In [7]:
import os, sys

# Go up THREE levels (project root directory)
project_root = os.path.dirname(os.path.dirname(os.path.dirname(os.getcwd())))

# Append the new path to sys.path
if project_root not in sys.path:
    sys.path.append(project_root)
    print("Project root added to sys.path")
else:
    print("Project root already in sys.path")

Project root already in sys.path


2. **Module 1:** `gro_processing`
    - `read_gro` — Turn `.gro` files into `DataFrame` (unsorted).
    - `dataframe_gro` — Uses `DataFrame` by `'read_gro'` to created multi-index `DataFrame` (sorted).

In [10]:
# Import module
from utils import gro_processing as gp

# Data directory can be accessed due to root PATH we set previously
file = os.path.join(project_root, 'data/npt-HK4.gro')

# Extracts data from .gro file into DataFrame (unsorted)
data, title, num_atoms, box_dimensions = gp.read_gro(file) 

# Parameter
box_length = box_dimensions[0]

# Create multi-index DataFrame (sorted)
df_gro = gp.dataframe_gro(data, box_length, positions=True, velocities=False, oxygen_midpoints=True)

# display(data)
# display(df_gro)

3. **Module 2:** `oxygen_midpoint`
    - More to a _helper module_ for `gro_processing`.
    - `compute_midpoints_df_jit` — Uses `DataFrame` by `'read_gro'` to create single-index `Dataframe` with only `res_id` and oxygen midpoints (x,y,z).

In [11]:
from utils import oxygen_midpoint as om

# Extract box length from box_dimensions string 
box_length = box_dimensions[0]

# Creates DataFrame with oxygen atom midpoints
midpoints_df_jit = om.compute_midpoints_df_jit(data, box_length)

# display(midpoints_df_jit)

---

# 1. Setting modules PATH

Setting modules PATH to access python packages (**import resolution**). 
- We need an **absolute path** to the **root** of the project (i.e. parent of the `utils` package).
- This way, our absolute imports (i.e. `from utils import oxygen_midpoints`) **knows to search** starting from the root of the project.

Adjust according to the level of nesting of `.ipynb` file.

In [1]:
import os, sys

# Go up THREE levels (project root directory)
project_root = os.path.dirname(os.path.dirname(os.path.dirname(os.getcwd())))

# Append the new path to sys.path
if project_root not in sys.path:
    sys.path.append(project_root)
    print("Project root added to sys.path")
else:
    print("Project root already in sys.path")
    
# Go up N levels (general case)
# def go_up_n_dirs(current_path, n):
#     for _ in range(n):
#         current_path = os.path.dirname(current_path)
#     return current_path
# project_root = go_up_n_dirs(os.getcwd(), 3)

Project root added to sys.path


In [3]:
# Checking
# sys.path

> Note that `sys.path` is only for **import resolution** (finding packages/modules). 
> 
> Trying to link to a file or directory does not work :x:, i.e. `file = ../../utils/blabla`.
> 
> But since we already have the root PATH, can just:
> ```python
> # basically (project_root + '/data/npt-HK4.gro')
> file = os.path.join(project_root, 'data/npt-HK4.gro')
> ```

---

# 2. Testing utils: `oxygen_midpoint.py`

## Function 1: `minimum_image_jit`

On first run, import takes longer time because of compilation from high-level code (`.py`) $\longrightarrow$ bytecode (`.pyc`) and Numba-bytecode (`.nbc`)

The two files formats, `.pyc` and `.nbc` are stored in `__pycache__` that is within `utils` directory. After stored, any time those functions are called, it is faster because skips compilation step and **directly uses cached bytecode**.

In [3]:
# Import module
from utils import oxygen_midpoint as om

# Parameters
dx = 2
box_length = 5
reciprocal_half_box = 1.0 / (0.5 * box_length) 

# First run will be slower due to compilation
# Testing (success)
om.minimum_image_jit(dx, box_length, reciprocal_half_box)

2.0

Testing wildcard functionality of `utils/__init__.py`. 

Wildcards can be defined in `__init__` or `<modules>.py`. [More about init and wildcards](https://www.youtube.com/watch?v=EH-TFaX-R-o)

In [ ]:
# Imports everything inside __all__ (success)
from utils import *

# Testing (success)
oxygen_midpoint.minimum_image_jit(dx, box_length, reciprocal_half_box)

2.0

## Function 2: `compute_midpoints_df_jit`

This won't work until we process `data` below. Come back later

In [ ]:
# Extract box length from box_dimensions string 
box_length = box_dimensions[0]

# Testing (success)
# compute_midpoints_df_jit(data, box_length) # Eliminate overhead (NOT NECESSARY SINCE CACHED)
midpoints_df_jit = om.compute_midpoints_df_jit(data, box_length)
# %timeit _ = compute_midpoints_df_jit(data, box_length)

display(midpoints_df_jit)

,res_id,mid_x,mid_y,mid_z
0,1,1.55850,0.40900,11.16949
1,2,0.44150,0.65700,10.78750
2,3,0.76450,10.72000,0.89800
3,4,0.15751,0.97750,0.13500
4,5,5.54400,0.11051,5.77600
...,...,...,...,...
1496,1497,8.55200,6.78500,8.62900
1497,1498,8.60450,0.93600,7.08450
1498,1499,5.25950,0.52650,11.24649
1499,1500,5.92950,10.45100,3.18150


---

# 3. Testing utils: `gro_processing.py`

## Function 1: `read_gro`

Able to read any `.gro` file and convert into `DataFrame`.
- Fixed issue where `'atom_id'` resets to $0$ after the $99,999^{\text{th}}$ atom.
- Made sure after the $199,999^{\text{th}}$ atom, continue counting from $200,000$ atom onwards.

In [5]:
# Import module
from utils import gro_processing as gp

# Data directory can be accessed due to root PATH we set previously
file = os.path.join(project_root, 'data/npt-HK4.gro')

In [6]:
# Testing (success)
data, title, num_atoms, box_dimensions = gp.read_gro(file) 

display(data)
print(f"Title: {title}")
print(f"Number of atoms: {num_atoms}")
print(f"Box dimensions: {box_dimensions}")

,res_id,res_name,atom_name,atom_id,x,y,z,Vx,Vy,Vz
0,1,HK4,H28,1,1.602,0.962,0.912,-0.0935,0.8795,-2.1573
1,1,HK4,C48,2,1.565,0.879,0.852,-0.2814,-0.6983,0.0954
2,1,HK4,C47,3,1.655,0.809,0.773,0.4154,0.1398,0.1445
3,1,HK4,H27,4,1.754,0.855,0.756,0.1277,0.7999,0.2091
4,1,HK4,C46,5,1.603,0.717,0.683,-0.0655,0.7587,-0.2231
...,...,...,...,...,...,...,...,...,...,...
126079,1501,HK4,H23,126080,0.953,5.984,1.494,-1.2440,0.1068,-0.7226
126080,1501,HK4,C44,126081,0.736,6.009,1.525,0.1513,0.5200,0.5632
126081,1501,HK4,H24,126082,0.707,6.041,1.425,0.3100,2.4172,1.1123
126082,1501,HK4,C45,126083,0.632,5.988,1.615,-0.0110,-0.1637,0.2235


Title: mol only system in water
Number of atoms: 126084
Box dimensions: [11.24798 11.24798 11.24798]


## Function 2: `dataframe_gro`

Outputs **multi-index** `DataFrame` with atom datas.
- Able to include or exclude atom's data: `positions` , `velocities` and `oxygen_midpoints`.

In [ ]:
box_length = box_dimensions[0]

# Testing (success)
df_gro = gp.dataframe_gro(data, box_length, positions=True, velocities=False, oxygen_midpoints=True)
display(df_gro)

res_name  atom_id      x      y      z  midO_x  midO_y  \
res_id atom_name                                                          
1      H28            HK4        1  1.602  0.962  0.912  1.5585   0.409   
       C48            HK4        2  1.565  0.879  0.852  1.5585   0.409   
       C47            HK4        3  1.655  0.809  0.773  1.5585   0.409   
       H27            HK4        4  1.754  0.855  0.756  1.5585   0.409   
       C46            HK4        5  1.603  0.717  0.683  1.5585   0.409   
...                   ...      ...    ...    ...    ...     ...     ...   
1501   H23            HK4   126080  0.953  5.984  1.494  0.2115   5.783   
       C44            HK4   126081  0.736  6.009  1.525  0.2115   5.783   
       H24            HK4   126082  0.707  6.041  1.425  0.2115   5.783   
       C45            HK4   126083  0.632  5.988  1.615  0.2115   5.783   
       H25            HK4   126084  0.542  6.051  1.607  0.2115   5.783   

                    midO_z  
res_id atom_name            
1      H28        11.16949  
       C48        11.16949  
       C47        11.16949  
       H27        11.16949  
       C46        11.16949  
...                    ...  
1501   H23         1.27700  
       C44         1.27700  
       H24         1.27700  
       C45         1.27700  
       H25         1.27700  

[126084 rows x 8 columns]